In [ ]:
!git clone https://github.com/HIMASREE-2102/Image-Caption-Generator-CNN-LSTM.git

In [ ]:
%cd /content/Image-Caption-Generator-CNN-LSTM

In [ ]:
!ls -lah

In [ ]:
from google.colab import files

files.download('/content/Image-Caption-Generator-CNN-LSTM/imagecaptiongenerator.ipynb')

In [ ]:
import json

path = "/content/Image-Caption-Generator-CNN-LSTM/imagecaptiongenerator.ipynb"

with open(path, "r") as f:
    nb = json.load(f)

print("Total cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"][:15]):
    print(f"\n{'='*50}")
    print("CELL:", i, "| TYPE:", cell["cell_type"])
    print("".join(cell["source"])[:1200])

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list -s flickr8k

In [ ]:
!kaggle datasets download -d adityajn105/flickr8k

In [ ]:
!unzip -q flickr8k.zip

In [ ]:
import os

print("Images directory exists:", os.path.exists("Images"))
print("Captions file exists:", os.path.exists("captions.txt"))

if os.path.exists("Images"):
    print("Number of images:", len(os.listdir("Images")))

In [ ]:
for i, cell in enumerate(nb["cells"][15:], start=15):
    print(f"\n{'='*50}")
    print(f"CELL {i} | {cell['cell_type']}")
    print("".join(cell["source"])[:2000])

In [ ]:
!pip -q install transformers sentencepiece accelerate gradio

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

caption_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

print("BLIP model loaded successfully!")

In [ ]:
import gradio as gr

def generate_caption(image):
    if image is None:
        return "Please upload an image"

    inputs = processor(
        image.convert("RGB"),
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = caption_model.generate(
            **inputs,
            num_beams=5,
            max_new_tokens=30
        )

    return processor.decode(
        output[0],
        skip_special_tokens=True
    )

ui = gr.Interface(
    fn=generate_caption,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Textbox(label="Generated Caption"),
    title="AI Image Caption Generator",
    description="Upload an image to generate an AI caption."
)

ui.launch(share=True)